# Cross-Model Shared Keyword SPAN Analysis

Two-part analysis:

**Part 1 — Shared Keyword SPAN:** Compute SPAN for ALL keywords shared across ≥ 3 models.
- `shared_keyword_span.csv` — every shared keyword × every model
- `shared_keyword_span_summary.csv` — per-model avg-SPAN summary

**Part 2 — Trend Category SPAN:** From the shared keywords, classify the top-50 in each trend
category (Emerging / Stable / Decaying), then compute their SPAN per model.
- `trend_category_keywords.csv` — the 150 classified keywords
- `trend_category_span.csv` — SPAN per keyword × model with category
- `trend_category_span_summary.csv` — per model × category avg-SPAN

In [1]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")

## Configuration

In [2]:
BASE = Path("/home/nedo/Kuliah/TA/Program")
DATA_DIR = BASE / "data" / "preprocess"
RESULTS_DIR = BASE / "results"

LIST_SUBJECT = ["cs", "math", "physics"]
MODELS = ["dtm", "lda", "top2vec", "bertopic", "topicGpt"]
MODEL_LABELS = {
    "dtm": "DTM", "lda": "LDA", "top2vec": "Top2Vec",
    "bertopic": "BERTopic", "topicGpt": "TopicGPT",
}

MIN_MODELS = 3        # keyword must appear in >= 3 models
TOP_K = 50            # top keywords per trend category (Part 2)

for subject in LIST_SUBJECT:
    (RESULTS_DIR / "shared" / "tren" / subject).mkdir(parents=True, exist_ok=True)

print(f"Models: {list(MODEL_LABELS.values())}")
print(f"Min models for keyword inclusion: {MIN_MODELS}")
print(f"Top-K per trend category: {TOP_K}")
print(f"Subjects: {LIST_SUBJECT}")

Models: ['DTM', 'LDA', 'Top2Vec', 'BERTopic', 'TopicGPT']
Min models for keyword inclusion: 3
Top-K per trend category: 50
Subjects: ['cs', 'math', 'physics']


## Helper Functions

In [3]:
def compute_corpus_word_freq(subject):
    """Count each word across all documents in corpus (v̂ₖ)."""
    df = pd.read_csv(DATA_DIR / subject / "bow/v1.csv")
    word_freq = defaultdict(int)
    for text_val in df["text"]:
        try:
            tokens = ast.literal_eval(text_val)
            if isinstance(tokens, list):
                for w in tokens:
                    word_freq[w] += 1
        except (ValueError, SyntaxError):
            for w in str(text_val).split():
                word_freq[w] += 1
    return word_freq


def load_topic_words_by_year(model, subject):
    """Load topic-word evolution → {year: [set of words, ...]}."""
    evo_path = RESULTS_DIR / model / "temporal" / subject / "topic_word_evolution.csv"
    if not evo_path.exists():
        return {}, []
    evo_df = pd.read_csv(evo_path)
    years = sorted(evo_df["year"].unique())
    topic_words_by_year = defaultdict(list)
    for _, row in evo_df.iterrows():
        words = set(w.strip() for w in str(row["top_words"]).split(","))
        topic_words_by_year[int(row["year"])].append(words)
    return topic_words_by_year, years


def compute_span(keyword, topic_words_by_year, years):
    """SPAN = longest consecutive years keyword appears in any topic."""
    trend = []
    for y in years:
        found = any(keyword in words for words in topic_words_by_year.get(y, []))
        trend.append(1 if found else 0)
    max_span = 0
    current = 0
    for t in trend:
        if t == 1:
            current += 1
            max_span = max(max_span, current)
        else:
            current = 0
    return max_span, trend

---

# Part 1: Shared Keyword SPAN

Compute SPAN for **ALL** keywords shared across ≥ 3 models.

### 1.1 Collect Shared Terms (≥ 3 Models)

In [4]:
all_model_data = {}   # {subject: {model: (tw_by_year, years)}}
all_shared_keywords = {}  # {subject: [sorted list of words]}
all_corpus_freq = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Collecting terms: {subject.upper()}")
    print(f"{'='*70}")

    model_data = {}
    model_terms = defaultdict(set)

    for model in MODELS:
        tw, years = load_topic_words_by_year(model, subject)
        model_data[model] = (tw, years)
        for year_words in tw.values():
            for words_in_topic in year_words:
                model_terms[model].update(words_in_topic)
        print(f"  {MODEL_LABELS[model]:>10s}: {len(model_terms[model]):>6,} unique terms")

    # Count how many models discovered each term
    term_counts = defaultdict(int)
    for model, terms in model_terms.items():
        for term in terms:
            term_counts[term] += 1

    shared_terms = {term for term, count in term_counts.items() if count >= MIN_MODELS}
    print(f"\n  Terms in >= {MIN_MODELS} models: {len(shared_terms):,}")

    # Compute corpus frequency
    print(f"  Computing corpus word frequencies...")
    corpus_freq = compute_corpus_word_freq(subject)
    print(f"  Corpus vocabulary: {len(corpus_freq):,} unique words")

    # Sort by corpus frequency (descending)
    candidate_freqs = [(w, corpus_freq.get(w, 0)) for w in shared_terms]
    candidate_freqs.sort(key=lambda x: -x[1])
    shared_keywords = [w for w, _ in candidate_freqs]

    all_model_data[subject] = model_data
    all_shared_keywords[subject] = shared_keywords
    all_corpus_freq[subject] = corpus_freq


         DTM:    550 unique terms
         LDA:  2,748 unique terms
     Top2Vec:  7,362 unique terms
    BERTopic: 10,879 unique terms
    TopicGPT:  3,357 unique terms

  Terms in >= 3 models: 2,574
  Computing corpus word frequencies...
  Corpus vocabulary: 146,603 unique words

         DTM:    251 unique terms
         LDA:  1,319 unique terms
     Top2Vec:  5,093 unique terms
    BERTopic:  6,831 unique terms
    TopicGPT:  1,526 unique terms

  Terms in >= 3 models: 1,281
  Computing corpus word frequencies...
  Corpus vocabulary: 100,004 unique words

         DTM:    288 unique terms
         LDA:  1,450 unique terms
     Top2Vec:  5,866 unique terms
    BERTopic: 11,794 unique terms
    TopicGPT:  2,292 unique terms

  Terms in >= 3 models: 1,724
  Computing corpus word frequencies...
  Corpus vocabulary: 122,859 unique words


### 1.2 Compute SPAN for All Shared Keywords

In [5]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Shared Keyword SPAN: {subject.upper()}")
    print(f"{'='*70}")

    model_data = all_model_data[subject]
    corpus_freq = all_corpus_freq[subject]
    shared_keywords = all_shared_keywords[subject]
    total_shared = len(shared_keywords)
    ref_years = model_data["dtm"][1]

    print(f"  Shared keywords: {total_shared:,}")
    print(f"  Years: {ref_years[0]}–{ref_years[-1]} ({len(ref_years)} years)")

    # Compute SPAN for each keyword × each model
    rows = []
    for word in shared_keywords:
        v_hat = corpus_freq.get(word, 0)
        row = {"word": word, "v_hat": v_hat}

        for model in MODELS:
            tw, years = model_data[model]
            span, trend = compute_span(word, tw, years)
            s_dict = span / v_hat if v_hat > 0 else 0.0
            label = MODEL_LABELS[model]
            row[f"span_{label}"] = span
            row[f"trend_{label}"] = str(trend)
            row[f"s_dict_{label}"] = round(s_dict, 6)

        rows.append(row)

    result_df = pd.DataFrame(rows)

    # Save shared_keyword_span.csv
    out_dir = RESULTS_DIR / "shared" / "tren" / subject
    result_df.to_csv(out_dir / "shared_keyword_span.csv", index=False)

    # Summary: avg-SPAN per model
    summary_rows = []
    for model in MODELS:
        label = MODEL_LABELS[model]
        spans = result_df[f"span_{label}"]
        s_dicts = result_df[f"s_dict_{label}"]

        n_captured = int((spans > 0).sum())
        captured_mask = spans > 0

        avg_span_paper = s_dicts[captured_mask].mean() if n_captured > 0 else 0.0
        avg_span_simple = spans[captured_mask].mean() if n_captured > 0 else 0.0

        summary_rows.append({
            "model": label,
            "n_keywords": total_shared,
            "n_captured": n_captured,
            "capture_pct": round(n_captured / total_shared * 100, 2),
            "avg_span_paper": round(avg_span_paper, 8),
            "avg_span_simple": round(avg_span_simple, 4),
            "sum_s_dict": round(s_dicts.sum(), 6),
            "max_span": int(spans.max()),
            "n_full_span": int((spans == len(ref_years)).sum()),
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(out_dir / "shared_keyword_span_summary.csv", index=False)

    # Print results
    print(f"\n  {'Model':<10s} {'avg-SPAN(paper)':>16s} {'avg-SPAN(simple)':>16s} {'Captured':>12s} {'Full':>5s}")
    print(f"  {'-'*65}")
    for _, s in summary_df.iterrows():
        print(f"  {s['model']:<10s} {s['avg_span_paper']:>16.8f} {s['avg_span_simple']:>16.4f} "
              f"{s['n_captured']:>5d} ({s['capture_pct']:>5.1f}%) {s['n_full_span']:>5d}")

    # Top-10 keywords
    print(f"\n  Top 10 keywords (by corpus freq):")
    span_cols = [f"span_{MODEL_LABELS[m]}" for m in MODELS]
    print(f"  {'Word':<20s} {'v̂ₖ':>8s}  " + "  ".join(f"{MODEL_LABELS[m]:>8s}" for m in MODELS))
    for _, r in result_df.head(10).iterrows():
        spans_str = "  ".join(f"{r[c]:>8d}" for c in span_cols)
        print(f"  {r['word']:<20s} {r['v_hat']:>8d}  {spans_str}")

    print(f"\n  Saved: {out_dir / 'shared_keyword_span.csv'}")
    print(f"  Saved: {out_dir / 'shared_keyword_span_summary.csv'}")


Shared Keyword SPAN: CS
  Shared keywords: 2,574
  Years: 2000–2025 (26 years)

  Model       avg-SPAN(paper) avg-SPAN(simple)     Captured  Full
  -----------------------------------------------------------------
  DTM              0.00070024           6.7336   428 ( 16.6%)    14
  LDA              0.21522520           5.3654  1661 ( 64.5%)    64
  Top2Vec          0.24985222           6.6541  2498 ( 97.0%)   102
  BERTopic         0.18402646           5.1711  2530 ( 98.3%)    12
  TopicGPT         0.17432613           4.7734  1840 ( 71.5%)    14

  Top 10 keywords (by corpus freq):
  Word                      v̂ₖ       DTM       LDA   Top2Vec  BERTopic  TopicGPT
  algorithm               81182        26        26        26        21        23
  network                 80079        26        25        25        20        25
  image                   76073        17        24        24        18        20
  dataset                 65819        13        15         7         3         

---

# Part 2: Trend Category SPAN

From the shared keywords (Part 1), classify the **top-50** per trend category
(Emerging / Stable / Decaying) using TF-IDF slope, then compute their SPAN per model.

### 2.1 Classify Shared Keywords into Trend Categories

In [6]:
def compute_yearly_tfidf(subject):
    """Compute average TF-IDF score per word per year from raw documents."""
    df = pd.read_csv(DATA_DIR / subject / "emb/v1.csv")
    df["year"] = pd.to_datetime(df["submitted_date"]).dt.year

    def to_text(val):
        try:
            tokens = ast.literal_eval(val)
            if isinstance(tokens, list):
                return " ".join(tokens)
        except (ValueError, SyntaxError):
            pass
        return str(val)

    df["text_str"] = df["text"].apply(to_text)
    years = sorted(df["year"].unique())

    yearly_scores = {}
    for year in years:
        year_docs = df[df["year"] == year]["text_str"].tolist()
        if len(year_docs) < 5:
            continue
        tfidf = TfidfVectorizer(max_features=5000, min_df=2, stop_words="english")
        matrix = tfidf.fit_transform(year_docs)
        feature_names = tfidf.get_feature_names_out()
        avg_scores = np.asarray(matrix.mean(axis=0)).flatten()
        for word, score in zip(feature_names, avg_scores):
            if word not in yearly_scores:
                yearly_scores[word] = {}
            yearly_scores[word][year] = float(score)

    return yearly_scores, years


def classify_keywords_from_pool(yearly_scores, years, candidate_words, top_k=50):
    """
    Classify words using LINEAR REGRESSION SLOPE of TF-IDF over time.
    Only considers words in candidate_words.
    Returns top_k per category: emerging, stable, decaying.
    """
    word_stats = []
    years_arr = np.array(years, dtype=float)

    for word in candidate_words:
        scores = yearly_scores.get(word, {})
        vals = np.array([scores.get(y, 0.0) for y in years])

        n_present = np.sum(vals > 0)
        if n_present < 3:
            continue

        slope, intercept, r_val, p_val, std_err = linregress(years_arr, vals)
        overall_avg = vals.mean()

        word_stats.append({
            "word": word,
            "slope": slope,
            "overall_avg": overall_avg,
        })

    stats_df = pd.DataFrame(word_stats)

    # Emerging: highest positive slope
    emerging_pool = stats_df[stats_df["slope"] > 0]
    emerging = emerging_pool.nlargest(top_k, "slope")

    # Stable: high avg TF-IDF + lowest absolute slope
    used = set(emerging["word"])
    stable_pool = stats_df[~stats_df["word"].isin(used)].copy()
    stable_pool["stability"] = stable_pool["overall_avg"] / (
        1 + stable_pool["slope"].abs() * 10000
    )
    stable = stable_pool.nlargest(top_k, "stability")

    # Decaying: strongest negative slope
    used.update(stable["word"])
    decay_pool = stats_df[
        (~stats_df["word"].isin(used)) & (stats_df["slope"] < 0)
    ]
    decaying = decay_pool.nsmallest(top_k, "slope")

    return emerging, stable, decaying


# Classify for each subject
keyword_cache = {}  # {subject: (emerging_df, stable_df, decaying_df)}

for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"TF-IDF Classification: {subject.upper()}")
    print(f"{'='*70}")

    print(f"  Computing yearly TF-IDF...")
    yearly_scores, years = compute_yearly_tfidf(subject)
    print(f"  {len(yearly_scores):,} unique words tracked")

    # Classify only from shared keyword pool
    candidates = set(all_shared_keywords[subject])
    emerging, stable, decaying = classify_keywords_from_pool(
        yearly_scores, years, candidates, top_k=TOP_K
    )
    keyword_cache[subject] = (emerging, stable, decaying)

    # Save trend_category_keywords.csv
    out_dir = RESULTS_DIR / "shared" / "tren" / subject
    kw_rows = []
    for cat, cat_df in [("emerging", emerging), ("stable", stable), ("decaying", decaying)]:
        for _, r in cat_df.iterrows():
            kw_rows.append({
                "word": r["word"],
                "category": cat,
                "slope": round(r["slope"], 8),
                "overall_avg": round(r["overall_avg"], 8),
            })
    kw_df = pd.DataFrame(kw_rows)
    kw_df.to_csv(out_dir / "trend_category_keywords.csv", index=False)

    for cat, icon, cat_df in [
        ("Emerging", "📈", emerging),
        ("Stable", "🔒", stable),
        ("Decaying", "📉", decaying),
    ]:
        print(f"\n  {icon} {cat} ({len(cat_df)} keywords):")
        print(f"  {'Word':25s} {'Slope':>12s} {'Avg TF-IDF':>12s}")
        print(f"  {'-'*50}")
        for _, r in cat_df.head(10).iterrows():
            print(f"  {r['word']:25s} {r['slope']:12.6f} {r['overall_avg']:12.6f}")
        if len(cat_df) > 10:
            print(f"  ... ({len(cat_df) - 10} more)")

    print(f"\n  Saved: {out_dir / 'trend_category_keywords.csv'}")


TF-IDF Classification: CS
  Computing yearly TF-IDF...
  12,598 unique words tracked

  📈 Emerging (50 keywords):
  Word                             Slope   Avg TF-IDF
  --------------------------------------------------
  learning                      0.000928     0.015563
  deep                          0.000662     0.005554
  training                      0.000656     0.006921
  models                        0.000630     0.013371
  image                         0.000608     0.008584
  neural                        0.000560     0.007821
  dataset                       0.000544     0.004424
  datasets                      0.000485     0.004442
  detection                     0.000456     0.007047
  tasks                         0.000451     0.005846
  ... (40 more)

  🔒 Stable (50 keywords):
  Word                             Slope   Avg TF-IDF
  --------------------------------------------------
  using                         0.000051     0.015071
  analysis                     -0.

### 2.2 SPAN per Category Keyword × Model

In [7]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Trend Category SPAN: {subject.upper()}")
    print(f"{'='*70}")

    model_data = all_model_data[subject]
    corpus_freq = all_corpus_freq[subject]
    emerging, stable, decaying = keyword_cache[subject]
    ref_years = model_data["dtm"][1]

    # Combine all classified keywords
    all_keywords = []
    for cat, cat_df in [("emerging", emerging), ("stable", stable), ("decaying", decaying)]:
        for _, r in cat_df.iterrows():
            all_keywords.append((r["word"], cat))

    print(f"  Total classified keywords: {len(all_keywords)}")

    # Compute SPAN for each keyword × model
    rows = []
    for word, category in all_keywords:
        v_hat = corpus_freq.get(word, 0)
        row = {"word": word, "category": category, "v_hat": v_hat}

        for model in MODELS:
            tw, years = model_data[model]
            span, trend = compute_span(word, tw, years)
            s_dict = span / v_hat if v_hat > 0 else 0.0
            label = MODEL_LABELS[model]
            row[f"span_{label}"] = span
            row[f"trend_{label}"] = str(trend)
            row[f"s_dict_{label}"] = round(s_dict, 6)

        rows.append(row)

    cat_span_df = pd.DataFrame(rows)

    out_dir = RESULTS_DIR / "shared" / "tren" / subject
    cat_span_df.to_csv(out_dir / "trend_category_span.csv", index=False)

    # Print cross-model comparison per category
    span_cols = [f"span_{MODEL_LABELS[m]}" for m in MODELS]
    for cat, icon in [("emerging", "📈"), ("stable", "🔒"), ("decaying", "📉")]:
        cat_data = cat_span_df[cat_span_df["category"] == cat]
        print(f"\n  {icon} {cat.upper()} (top 10 of {len(cat_data)}):")
        hdr = f"  {'Word':<20s} {'v̂ₖ':>8s}  " + "  ".join(f"{MODEL_LABELS[m]:>8s}" for m in MODELS)
        print(hdr)
        for _, r in cat_data.head(10).iterrows():
            spans_str = "  ".join(f"{r[c]:>8d}" for c in span_cols)
            print(f"  {r['word']:<20s} {r['v_hat']:>8d}  {spans_str}")

    print(f"\n  Saved: {out_dir / 'trend_category_span.csv'}")


Trend Category SPAN: CS
  Total classified keywords: 150

  📈 EMERGING (top 10 of 50):
  Word                      v̂ₖ       DTM       LDA   Top2Vec  BERTopic  TopicGPT
  learning                35718        12        26        26        13        26
  deep                    22805        11        15        16        12        15
  training                45569        12        17        14         3        13
  models                      6         0        26        23         6        14
  image                   76073        17        24        24        18        20
  neural                  12037         9        24        24        15        16
  dataset                 65819        13        15         7         3         7
  datasets                   30         0        14         3         1         4
  detection               27087        12        18        18        17        15
  tasks                       2         0        14        17         9        15

  🔒 STABL

### 2.3 avg-SPAN Summary per Model × Category

| Metric | Formula | Description |
|--------|---------|-------------|
| **avg-SPAN (paper)** | (1/N_captured) × Σ (Sₖ/v̂ₖ) | Rewards rare-but-persistent terms |
| **avg-SPAN (simple)** | (1/N_captured) × Σ Sₖ | Simple mean of raw SPAN |

Both averages are computed **only over captured keywords** (SPAN > 0).

In [8]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Trend Category avg-SPAN: {subject.upper()}")
    print(f"{'='*70}")

    out_dir = RESULTS_DIR / "shared" / "tren" / subject
    cat_span_df = pd.read_csv(out_dir / "trend_category_span.csv")

    summary_rows = []

    for model in MODELS:
        label = MODEL_LABELS[model]
        spans_col = f"span_{label}"
        s_dict_col = f"s_dict_{label}"

        for cat in ["all", "emerging", "stable", "decaying"]:
            if cat == "all":
                subset = cat_span_df
            else:
                subset = cat_span_df[cat_span_df["category"] == cat]

            n_keywords = len(subset)
            spans = subset[spans_col]
            s_dicts = subset[s_dict_col]

            n_captured = int((spans > 0).sum())
            captured_mask = spans > 0

            avg_span_paper = s_dicts[captured_mask].mean() if n_captured > 0 else 0.0
            avg_span_simple = spans[captured_mask].mean() if n_captured > 0 else 0.0

            summary_rows.append({
                "model": label,
                "category": cat,
                "n_keywords": n_keywords,
                "n_captured": n_captured,
                "capture_pct": round(n_captured / n_keywords * 100, 2) if n_keywords > 0 else 0.0,
                "avg_span_paper": round(avg_span_paper, 8),
                "avg_span_simple": round(avg_span_simple, 4),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(out_dir / "trend_category_span_summary.csv", index=False)

    # Print: overall first
    print(f"\n  Overall (all categories):")
    print(f"  {'Model':<10s} {'avg-SPAN(paper)':>16s} {'avg-SPAN(simple)':>16s} {'Captured':>12s}")
    print(f"  {'-'*58}")
    overall = summary_df[summary_df["category"] == "all"]
    for _, s in overall.iterrows():
        print(f"  {s['model']:<10s} {s['avg_span_paper']:>16.8f} {s['avg_span_simple']:>16.4f} "
              f"{s['n_captured']:>5d} ({s['capture_pct']:>5.1f}%)")

    # Per-category breakdown
    for cat, icon in [("emerging", "📈"), ("stable", "🔒"), ("decaying", "📉")]:
        cat_summary = summary_df[summary_df["category"] == cat]
        n_kw = cat_summary["n_keywords"].iloc[0]
        print(f"\n  {icon} {cat.upper()} ({n_kw} keywords):")
        print(f"  {'Model':<10s} {'avg-SPAN(paper)':>16s} {'avg-SPAN(simple)':>16s} {'Captured':>12s}")
        print(f"  {'-'*58}")
        for _, s in cat_summary.iterrows():
            print(f"  {s['model']:<10s} {s['avg_span_paper']:>16.8f} {s['avg_span_simple']:>16.4f} "
                  f"{s['n_captured']:>5d} ({s['capture_pct']:>5.1f}%)")

    print(f"\n  Saved: {out_dir / 'trend_category_span_summary.csv'}")

print("\n✅ All done!")


Trend Category avg-SPAN: CS

  Overall (all categories):
  Model       avg-SPAN(paper) avg-SPAN(simple)     Captured
  ----------------------------------------------------------
  DTM              0.00069311          12.1379    87 ( 58.0%)
  LDA              0.74878529          16.1241   145 ( 96.7%)
  Top2Vec          0.62264953          16.1477   149 ( 99.3%)
  BERTopic         0.34006657          10.7034   145 ( 96.7%)
  TopicGPT         0.48423743          11.4632   136 ( 90.7%)

  📈 EMERGING (50 keywords):
  Model       avg-SPAN(paper) avg-SPAN(simple)     Captured
  ----------------------------------------------------------
  DTM              0.00050993          10.6000    30 ( 60.0%)
  LDA              0.91599657          16.6735    49 ( 98.0%)
  Top2Vec          0.92847560          16.0000    50 (100.0%)
  BERTopic         0.48968694          10.0000    49 ( 98.0%)
  TopicGPT         0.67187742          12.2292    48 ( 96.0%)

  🔒 STABLE (50 keywords):
  Model       avg-SPAN(p